In [ ]:
import kagglehub

# Understanding the Iris Dataset

The Iris dataset is one of the most famous and widely-used datasets in machine learning and statistics. It's hosted on Kaggle Hub, making it easily accessible for download and analysis. This dataset serves as an excellent introduction to classification problems because it's well-structured and contains meaningful patterns that demonstrate fundamental ML concepts.

## Dataset Structure

The dataset contains measurements from three different species of Iris flowers: Setosa, Versicolor, and Virginica. Each species has exactly 50 samples, giving you a balanced dataset with 150 total observations. For each flower sample, four key numerical features are recorded: sepal length, sepal width, petal length, and petal width (all measured in centimeters). These four features serve as your input variables (X) for training classification models.

## Linear Separability Challenge

What makes the Iris dataset particularly instructive is its **mixed linear separability**. One of the three species—typically Setosa—is **linearly separable** from the other two. This means you can draw a straight line (or hyperplane in higher dimensions) that perfectly separates Setosa samples from Versicolor and Virginica samples. This property makes it ideal for testing linear classification algorithms like Logistic Regression or linear SVM kernels.

However, the other two species—**Versicolor and Virginica—are not linearly separable** from each other. Their measurements overlap significantly in feature space, meaning no single straight line can perfectly divide them. This characteristic demonstrates the importance of using non-linear classification methods like RBF (Radial Basis Function) kernels, polynomial kernels, or ensemble methods like XGBoost to handle more complex decision boundaries. This mixed difficulty makes the Iris dataset perfect for comparing how different algorithms handle varying levels of classification complexity.

In [ ]:
path = kagglehub.dataset_download("uciml/iris")
print(f"Dataset downloaded and extracted to: {path}")

# Exploring the Dataset: Initial Data Investigation

When starting any machine learning project, the first critical step is to thoroughly understand your data. Before you can build effective models or make informed decisions about preprocessing and feature engineering, you need to know exactly what you're working with. This involves examining the structure, content, and characteristics of your dataset.

## Why Data Exploration Matters

Understanding your dataset upfront saves time and prevents costly mistakes later in the pipeline. By investigating the data structure, you can identify potential issues, understand the relationships between variables, and make informed choices about which features to include in your model. This foundational step is essential for every data scientist and machine learning engineer.

## Key Information to Extract

The initial data exploration should focus on several key aspects. First, examine **what columns are present** in your dataset—these represent your features and target variable. Understanding column names helps you grasp what each piece of data represents in the real world. Second, determine the **data types** of each column. Are they numerical (integers, floats) or categorical (strings, categories)? This is crucial because machine learning algorithms typically require numerical inputs, so categorical data will need encoding later.

Third, identify the **number of features and samples** in your dataset. Features are the input variables (columns) that your model will learn from, while samples are the individual observations (rows) in your dataset. Knowing these dimensions helps you understand the scale of your problem and whether you have sufficient data for training. A dataset with many samples but few features presents different challenges than one with few samples but many features.

## Practical Benefits

By systematically exploring these aspects—columns, data types, features, and samples—you establish a solid foundation for your entire machine learning workflow. This information guides decisions about data cleaning, feature selection, model choice, and hyperparameter tuning. It's an investment that pays dividends throughout your project.

In [ ]:
import pandas as pd
orgdata = pd.read_csv(f"{path}/Iris.csv")
data = orgdata.drop('Id', axis=1)  # Remove the Id column as it's not needed for analysis
print(data.head())

print(data.shape)
print(data.columns)
print(data.info())
print(data['Species'].value_counts())

Do we have any nulls or NaN data, they we should remove such data or use imputation to update missing values.

In [ ]:
data[data.isnull().any(axis=1)]

We need split data in to two parts - one for training the model and another one of testing/validating the trained model. In general, we should more train data so that model will learn from different features and get trained. Generally 20% of data will be used for testing and 80% is for training.

In [ ]:

from sklearn.model_selection import train_test_split

traing_size = 0.8
testing_size = 1 - traing_size # 20% is used for testing

# We separate the features (X) from the target variable (y). Here is target variable is "Species"
X = data.drop('Species', axis=1)
y = data['Species']

# We split the dataset into training and testing sets using the train_test_split function from scikit-learn. 
# We specify the test size and a random state for reproducibility.
# x_train, y_train will be used to train the model, while x_test, y_test will be used to evaluate its performance.

# random state is used to ensure that the split is reproducible, meaning that every time you run the code with the 
# same random state, you will get the same split of data into training and testing sets.
def split_data(X, y, testing_size, random_state=42):
    x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=testing_size, random_state=random_state)
    return x_train, x_test, y_train, y_test

x_train, x_test, y_train, y_test = split_data(X, y, testing_size)

As you all know machine learning models only deals with numerical data. So we need to encode string data type field (species) to numerical values

In [ ]:

def encode_labels(y_train, y_test):
    import sklearn.preprocessing as preprocessing

    le = preprocessing.LabelEncoder()

    # here we encode the target variable (y) for both training and testing sets. 
    # The fit_transform method is used on the training set to learn the encoding, 
    # and then the transform method is applied to the test set to ensure that the same encoding is used for both sets.

    y_train_encoded = le.fit_transform(y_train)
    y_test_encoded = le.transform(y_test)

    return y_train_encoded, y_test_encoded

y_train_encoded, y_test_encoded = encode_labels(y_train, y_test)

print(x_train.shape, y_train_encoded.shape)
print(x_test.shape, y_test_encoded.shape)


In [ ]:
# Before training the model, we need to plot the data to understand the relationships between the features 
# and the target variable. This will ensure that we have a good understanding of the data and can identify 
# any patterns or correlations that may exist.

# Now we need to see if the all features in the dataset are useful for the model or not. 
# We can use pairplot to visualize the relationships between the features and the target variable (Species).
# The pairplot will show scatter plots for each pair of features, colored by the species, and histograms on the diagonal.
# The diag_kind=None argument is used to disable the diagonal plots, which are not needed in this case.
# This visualization will help us to see if there are any clear separations between the species based on the features,
# and if any features are more informative than others for distinguishing between the species.

import matplotlib.pyplot as plt
import seaborn as sns
sns.pairplot(data, hue="Species", diag_kind=None)
plt.show()

This pairplot will show us the relationships between the features and the target variable (Species). 
We can see that all features are more useful for distinguishing between the different species.
I think we can use all features for training the model.

Before training the model, we need to plot the data to understand the relationships between the features 
and the target variable. This will ensure that we have a good understanding of the data and can identify 
any patterns or correlations that may exist.

The following plot clearly shows that all features play role in classifing the species. So we will not drop any feature.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Scatter plot of Sepal Length vs Sepal Width, colored by Species
plt.figure(figsize=(8, 6))
sns.scatterplot(x='SepalLengthCm', y='SepalWidthCm', hue='Species', data=data, palette='Set1')
plt.title('Sepal Length vs Sepal Width by Species')
plt.xlabel('Sepal Length (cm)')
plt.ylabel('Sepal Width (cm)')
plt.legend(title='Species')
plt.show()

In [ ]:
# Scatter plot of Petal Length vs Sepal Width, colored by Species
plt.figure(figsize=(8, 6))
sns.scatterplot(x='PetalLengthCm', y='SepalWidthCm', hue='Species', data=data, palette='Set1')
plt.title('Petal Length vs Sepal Width by Species')
plt.xlabel('Petal Length (cm)')
plt.ylabel('Sepal Width (cm)')
plt.legend(title='Species')
plt.show()

In [ ]:

# Heatmap to show the correlation between features. 
# Encode the target variable (Species) to numeric values for correlation calculation
data['Species'] = data['Species'].astype('category').cat.codes

sns.heatmap(data.corr(), annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Iris Dataset')
plt.show()

In [ ]:


import sklearn.linear_model as linear_model
import sklearn.metrics as metrics
import numpy as np
# evaluate_model function takes a trained model and the test data (x_test and y_test) as input,
# and it evaluates the model's performance by calculating various metrics such as accuracy, precision, recall, and F1 score. 
# Accuracy is the ratio of correctly predicted instances to the total instances, 
# Precision is the ratio of correctly predicted positive observations to the total predicted positives, 
# Recall is the ratio of correctly predicted positive observations to all observations in actual class, 
# F1 score is the weighted average of precision and recall.
# It also generates a classification report and a confusion matrix to visualize the performance of the model.
# confusion matrix is a table that is used to evaluate the performance of a classification model. 
# It shows the number of true positives, true negatives, false positives, and false negatives. 
# The classification report provides a summary of the precision, recall, F1 score, and support for each class 
# in the target variable.

def evaluate_model(model, x_test, y_test):
    # if model is logistic regression, then model doesnt have predict method, we need to use predict_proba method to get the 
    # predicted probabilities and then convert them to class labels
    if isinstance(model, linear_model.LogisticRegression):
        predict_proba = model.predict_proba(x_test)
        predict = np.argmax(predict_proba, axis=1)
    else:
        predict = model.predict(x_test)

    # Calculate evaluation metrics
    accuracy = metrics.accuracy_score(y_test, predict)
    precision = metrics.precision_score(y_test, predict, average='weighted')
    recall = metrics.recall_score(y_test, predict, average='weighted')
    f1_score = metrics.f1_score(y_test, predict, average='weighted')

    # Generate classification report and confusion matrix
    # confusion matrix is a table that is used to evaluate the performance of a classification model. 
    # It shows the number of true positives, true negatives, false positives, and false negatives. 
    # The classification report provides a summary of the precision, recall, F1 score, and support for each class 
    # in the target variable.
    confusion_matrix = metrics.confusion_matrix(y_test, predict)
    classification_report = metrics.classification_report(y_test, predict)

    print("\nClassification Report:")
    print(classification_report)

    metrics.ConfusionMatrixDisplay.from_predictions(y_test, predict)
    plt.title('Confusion Matrix for ' + model.__class__.__name__)
    plt.show()
    return accuracy, precision, recall, f1_score




In [ ]:
# Logistic Regression is a linear model used for classification tasks. It is used to predict the probability 
# of a binary outcome (1/0, Yes/No, True/False) based on one or more predictor variables (features).
# In the case of the Iris dataset, we have a multi-class classification problem (three classes of iris),
# but logistic regression can still be applied using a one-vs-rest (OvR) approach, where a separate binary classifier is trained 
# for each class against all other classes. The logistic_model function trains a logistic regression model on the training data and 
# evaluates its performance on the test data using the evaluate_model function defined earlier. 


def logistic_model():
    logistic_model = linear_model.LogisticRegression()

    logistic_model.fit(x_train, y_train_encoded)
    logistic_model.score(x_test, y_test_encoded)
    return evaluate_model(logistic_model, x_test, y_test_encoded)

In [ ]:
accuracy, precision, recall, f1_score = logistic_model()

In [ ]:
# K-Nearest Neighbors (KNN) is a non-parametric, instance-based learning algorithm used for classification 
# and regression tasks. In classification, KNN predicts the class of a data point based on the majority class among its k nearest neighbors 
# in the feature space and regression tasks. 
# KNN is a simple algorithm that can be effective for certain types of data, but it can also be computationally expensive and 
# may not perform well with high-dimensional data or imbalanced datasets. 
# 
# The knn_model function trains a KNN classifier on the training data and evaluates its performance on the test data using the 
# evaluate_model function defined earlier.


from sklearn.neighbors import KNeighborsClassifier
def knn_model():
    knn_model = KNeighborsClassifier(n_neighbors=3)

    knn_model.fit(x_train, y_train_encoded)
    knn_model.score(x_test, y_test_encoded)
    return evaluate_model(knn_model, x_test, y_test_encoded)

In [ ]:
knn_model()

In [ ]:
# SVM (Support Vector Machine) is a powerful supervised learning algorithm used for classification and regression tasks. 
# It works by finding the optimal hyperplane that best separates the classes in the feature space.
# SVM can use different kernel functions (linear, polynomial, radial basis function, etc.) to handle non-linear relationships between 
# features and the target variable.
# linear kernel is used when the data is linearly separable,
# while the RBF (Radial Basis Function) kernel is used when the data is not linearly separable
# polynomial kernel is used when the data is not linearly separable and has a polynomial relationship between features and target variable.

# The svm_model function trains an SVM classifier with the specified kernel on the training data and evaluates its performance on 
# the test data using the evaluate_model function defined earlier.

def svm_model(kernel):
    from sklearn import svm
    svm_model = svm.SVC(kernel=kernel)

    svm_model.fit(x_train, y_train_encoded)
    svm_model.score(x_test, y_test_encoded)
    return evaluate_model(svm_model, x_test, y_test_encoded)

In [ ]:
# kernel can be 'linear', 'rbf', or 'poly' depending on the type of decision boundary we want to create.
# linear kernel is used when the data is linearly separable, 
# rbf (Radial Basis Function) kernel is used for non-linear data,
# poly (polynomial) kernel is used when the data has polynomial relationships.

svm_model(kernel='linear')
svm_model(kernel='rbf')
svm_model(kernel='poly')

In [ ]:
# XGBoost classifier is an implementation of the gradient boosting algorithm, which is a powerful ensemble 
# learning method that combines multiple weak learners (usually decision trees) to create a strong predictive model.
# XGBoost is known for its efficiency and performance, and it has been widely used in machine learning competitions 
# and real-world applications.
# The xgboost_model function trains an XGBoost classifier on the training data and evaluates its performance on the test data using the 
# evaluate_model function defined earlier.

def xgboost_model():
    from xgboost import XGBClassifier
    xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')

    xgb_model.fit(x_train, y_train_encoded)
    xgb_model.score(x_test, y_test_encoded)
    return evaluate_model(xgb_model, x_test, y_test_encoded)

In [ ]:
xgboost_model()

In [ ]:
# MLP Classifier (Multi-layer Perceptron) is a type of feedforward artificial neural network that consists of 
# multiple layers of nodes (neurons)
# MLP can be used for classification tasks and is capable of learning complex patterns in the data.
# MLP Classifier is a type of feedforward artificial neural network that consists of multiple layers of nodes (neurons).
# MLP can be used for classification tasks and is capable of learning complex patterns in the data.
# The mlp_model function trains an MLP classifier on the training data and evaluates its performance on the test data using the 
# evaluate_model function defined earlier.

def mlp_model():
    from sklearn.neural_network import MLPClassifier
    mlp_model = MLPClassifier(max_iter=1000)

    mlp_model.fit(x_train, y_train_encoded)
    mlp_model.score(x_test, y_test_encoded)
    return evaluate_model(mlp_model, x_test, y_test_encoded)

In [ ]:
mlp_model()

In [ ]:
# Finally, we can compare the performance of all the models we trained (Logistic Regression, KNN, SVM with different kernels, XGBoost, and MLP)
# We can create a summary table to compare the accuracy, precision, recall, and F1 score of each model.
# We can use a pandas DataFrame to create a summary table that compares the performance of all the models we trained.

testing_size = 0.2
x_train, x_test, y_train, y_test = split_data(X, y, testing_size)
y_train_encoded, y_test_encoded = encode_labels(y_train, y_test)

accuracy_logistic_tz_20, precision_logistic_tz_20, recall_logistic_tz_20, f1_logistic_tz_20 = logistic_model()
accuracy_knn_tz_20, precision_knn_tz_20, recall_knn_tz_20, f1_knn_tz_20 = knn_model()   
accuracy_svm_linear_tz_20, precision_svm_linear_tz_20, recall_svm_linear_tz_20, f1_svm_linear_tz_20 = svm_model(kernel='linear')
accuracy_svm_rbf_tz_20, precision_svm_rbf_tz_20, recall_svm_rbf_tz_20, f1_svm_rbf_tz_20 = svm_model(kernel='rbf')
accuracy_svm_poly_tz_20, precision_svm_poly_tz_20, recall_svm_poly_tz_20, f1_svm_poly_tz_20 = svm_model(kernel='poly')
accuracy_xgboost_tz_20, precision_xgboost_tz_20, recall_xgboost_tz_20, f1_xgboost_tz_20 = xgboost_model()
accuracy_mlp_tz_20, precision_mlp_tz_20, recall_mlp_tz_20, f1_mlp_tz_20 = mlp_model()

In [ ]:
# plot the results for different models and testing sizes 0.2 
# We can use a bar chart to compare the accuracy, precision, recall, and F1 score of each model for the testing size of 0.2.    

plot_data_tz_20 = {
    'Model': ['Logistic Regression', 'KNN', 'SVM (Linear)', 'SVM (RBF)', 'SVM (Poly)', 'XGBoost', 'MLP Classifier'],
    'Accuracy': [accuracy_logistic_tz_20, accuracy_knn_tz_20, accuracy_svm_linear_tz_20, accuracy_svm_rbf_tz_20, accuracy_svm_poly_tz_20, accuracy_xgboost_tz_20, accuracy_mlp_tz_20],
    'Precision': [precision_logistic_tz_20, precision_knn_tz_20, precision_svm_linear_tz_20, precision_svm_rbf_tz_20, precision_svm_poly_tz_20, precision_xgboost_tz_20, precision_mlp_tz_20],
    'Recall': [recall_logistic_tz_20, recall_knn_tz_20, recall_svm_linear_tz_20, recall_svm_rbf_tz_20, recall_svm_poly_tz_20, recall_xgboost_tz_20, recall_mlp_tz_20],
    'F1 Score': [f1_logistic_tz_20, f1_knn_tz_20, f1_svm_linear_tz_20, f1_svm_rbf_tz_20, f1_svm_poly_tz_20, f1_xgboost_tz_20, f1_mlp_tz_20]
}
def plot_model_performance(plot_data):
    import matplotlib.pyplot as plt
    import numpy as np
    labels = plot_data['Model']
    accuracy = plot_data['Accuracy']
    precision = plot_data['Precision']
    recall = plot_data['Recall']    
    f1_score = plot_data['F1 Score']
    x = np.arange(len(labels))  # the label locations
    width = 0.2  # the width of the bars    
    fig, ax = plt.subplots(figsize=(12, 6))
    rects1 = ax.bar(x - width, accuracy, width, label='Accuracy')
    rects2 = ax.bar(x, precision, width, label='Precision') 
    rects3 = ax.bar(x + width, recall, width, label='Recall')
    rects4 = ax.bar(x + 2*width, f1_score, width, label='F1 Score')
    ax.set_xlabel('Model')
    ax.set_title('Model Performance Comparison (Testing Size = 0.2)')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45)
    ax.legend()
    fig.tight_layout()
    plt.show()


In [ ]:
# if you see all above models are giving 100% accuracy, then we can say that the dataset is very simple and 
# all the models are able to learn the patterns in the data perfectly.
# but I got a doubt that if we can get 100% accuracy on the test set, then it means that the model is 
# overfitting the training data and may not generalize well to new, unseen data.
# or is my code working fine and the dataset is just simple enough that all models can achieve perfect accuracy?
# to validate this, we can try to use a different random state for the train-test split and see if the accuracy 
# remains high.

# let me see if testing_size = 0.5 and training_size = 0.5, then we will have more data in the test set and 
# less data in the training set, which may lead to lower accuracy if the models are not able to learn well 
# from the smaller training set.

testing_size = 0.9
x_train, x_test, y_train, y_test = split_data(X, y, testing_size)
y_train_encoded, y_test_encoded = encode_labels(y_train, y_test)

accuracy_logistic_tz_80, precision_logistic_tz_80, recall_logistic_tz_80, f1_logistic_tz_80 = logistic_model()
accuracy_knn_tz_80, precision_knn_tz_80, recall_knn_tz_80, f1_knn_tz_80 = knn_model()
accuracy_svm_linear_tz_80, precision_svm_linear_tz_80, recall_svm_linear_tz_80, f1_svm_linear_tz_80 = svm_model(kernel='linear')
accuracy_svm_rbf_tz_80, precision_svm_rbf_tz_80, recall_svm_rbf_tz_80, f1_svm_rbf_tz_80 = svm_model(kernel='rbf')
accuracy_svm_poly_tz_80, precision_svm_poly_tz_80, recall_svm_poly_tz_80, f1_svm_poly_tz_80 = svm_model(kernel='poly')
accuracy_xgboost_tz_80, precision_xgboost_tz_80, recall_xgboost_tz_80, f1_xgboost_tz_80 = xgboost_model()
accuracy_mlp_tz_80, precision_mlp_tz_80, recall_mlp_tz_80, f1_mlp_tz_80 = mlp_model()

In [ ]:
# if you see with testing_size = 0.8, the accuracy of all models has dropped significantly, 
# which indicates that the models are not able to learn well from the smaller training set and 
# are likely overfitting the training data when the training set is larger.

# so there is nothing wrong with the code, the dataset is just simple enough that all models can 
# achieve perfect accuracy when there is enough training data, but when the training data is reduced, 
# the models struggle to learn and generalize well, leading to a drop in accuracy.

In [ ]:
plot_data_tz_80 = {
    'Model': ['Logistic Regression', 'KNN', 'SVM (Linear)', 'SVM (RBF)', 'SVM (Poly)', 'XGBoost', 'MLP Classifier'],
    'Accuracy': [accuracy_logistic_tz_80, accuracy_knn_tz_80, accuracy_svm_linear_tz_80, accuracy_svm_rbf_tz_80, accuracy_svm_poly_tz_80, accuracy_xgboost_tz_80, accuracy_mlp_tz_80],
    'Precision': [precision_logistic_tz_80, precision_knn_tz_80, precision_svm_linear_tz_80, precision_svm_rbf_tz_80, precision_svm_poly_tz_80, precision_xgboost_tz_80, precision_mlp_tz_80],
    'Recall': [recall_logistic_tz_80, recall_knn_tz_80, recall_svm_linear_tz_80, recall_svm_rbf_tz_80, recall_svm_poly_tz_80, recall_xgboost_tz_80, recall_mlp_tz_80],
    'F1 Score': [f1_logistic_tz_80, f1_knn_tz_80, f1_svm_linear_tz_80, f1_svm_rbf_tz_80, f1_svm_poly_tz_80, f1_xgboost_tz_80, f1_mlp_tz_80]
}

plot_model_performance(plot_data_tz_80)